<a href="https://colab.research.google.com/github/Ndryl/.sample/blob/main/01%20-%20Language%20Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelos de Linguagem

Um modelo de linguagem causal recebe uma sequência de inteiros e devolve, para cada posição, uma pontuação para cada item do vocabulário. O texto existe apenas nas bordas do sistema: o tokenizador traduz caracteres em ids, e uma regra de decodificação converte pontuações em uma escolha. Gerar texto é repetir esse ciclo, um token por vez.

O notebook percorre esse caminho uma vez, detalha a tokenização, a leitura dos logits e os controles de geração, compara um modelo base com um modelo ajustado para instrução, e termina reunindo tudo em uma classe `LLM`.

In [1]:
# No Google Colab, use Ambiente de execução > GPU; nada precisa ser instalado.

import time

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

## Predição do próximo token

Quatro etapas separam um texto de entrada do próximo pedaço de texto: codificar, executar o forward pass, escolher um id e decodificar. Nesta seção elas aparecem uma vez, sem justificativa dos argumentos, para fixar o formato de cada resultado intermediário.

As duas células seguintes baixam o tokenizador e os pesos de um modelo base, treinado apenas para continuar texto. O download é da ordem de um gigabyte e começa aqui, de modo a ocorrer em paralelo com o restante da exposição.

In [3]:
BASE_NAME = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(BASE_NAME)
print(type(tokenizer).__name__)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Qwen2Tokenizer


In [4]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
base_model.eval()
print(type(base_model).__name__)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Qwen2ForCausalLM


A codificação traduz o texto em ids, que constituem a única entrada aceita pelo modelo.

In [7]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
print(inputs["input_ids"])

tensor([[ 785, 6722,  315, 9625,  374]])


O forward pass recebe os ids e devolve os logits. A saída da última posição corresponde à continuação do texto.

In [8]:
with torch.no_grad():
    outputs = base_model(**inputs)
last_logits = outputs.logits[0, -1]
print(last_logits.shape)

torch.Size([151936])


São mais de cento e cinquenta mil valores, um por item do vocabulário. A terceira etapa seleciona um deles, e a regra mais simples toma o de maior pontuação. A quarta traduz o id de volta em texto.

In [9]:
next_id = int(last_logits.argmax())
print(next_id, repr(tokenizer.decode([next_id])))

12095 ' Paris'


Esse é o ciclo completo. As próximas seções destrincham cada etapa, a partir das perguntas que a execução acima deixa em aberto: o que o tokenizador fez com a frase, o que existe nos logits além do maior valor, e como se passa de um token isolado para um texto inteiro.

## Tokenização

O tokenizador ocupa um conjunto pequeno de arquivos, distribuído junto do modelo e independente dos pesos. Ele contém o vocabulário e as regras que dividem o texto em pedaços, cada um associado a um id. Duas partes do vocabulário importam aqui: os pedaços aprendidos sobre o corpus e os marcadores reservados, acrescentados depois pelo autor do modelo.

### Vocabulário e segmentação

A codificação de uma frase produz a lista de ids que o modelo recebe.

In [10]:
text = "Agentes de IA precisam de ferramentas."
token_ids = tokenizer.encode(text)
print(token_ids)

[16810, 288, 409, 43090, 20234, 309, 409, 57039, 2838, 300, 13]


Cada id tem duas representações úteis: a forma bruta registrada no vocabulário e a forma decodificada.

In [11]:
pd.DataFrame(
    {
        "id": token_ids,
        "token": tokenizer.convert_ids_to_tokens(token_ids),
        "piece": [tokenizer.decode([token_id]) for token_id in token_ids],
    }
)

,id,token,piece
0,16810,Agent,Agent
1,288,es,es
2,409,Ġde,de
3,43090,ĠIA,IA
4,20234,Ġprecis,precis
5,309,am,am
6,409,Ġde,de
7,57039,Ġferr,ferr
8,2838,ament,ament
9,300,as,as


Na coluna bruta, `Ġ` representa o espaço. O espaço integra o token seguinte, com a consequência de que a mesma palavra recebe ids diferentes conforme a posição na frase, a caixa e a flexão.

Este tokenizador aplica BPE sobre bytes. O algoritmo parte de bytes isolados e funde iterativamente os pares mais frequentes do corpus até atingir o tamanho do vocabulário. Sequências frequentes convergem para um único token e sequências raras permanecem fragmentadas.

In [12]:
for variant in ["ferramenta", " ferramenta", "Ferramenta", " ferramentas"]:
    print(repr(variant), tokenizer.encode(variant))

'ferramenta' [69, 615, 2838, 64]
' ferramenta' [57039, 2838, 64]
'Ferramenta' [37, 615, 2838, 64]
' ferramentas' [57039, 2838, 300]


Números longos raramente ocorrem inteiros no corpus e acabam segmentados em grupos de poucos dígitos, o que contribui para erros de aritmética. Como a base são bytes, qualquer sequência é representável, e este vocabulário dispensa token de desconhecido: palavras inventadas, outros alfabetos e emojis recebem segmentação progressivamente mais fina.

In [13]:
for sample in ["1234567", "1.234.567", "antidesestabelecimento", "🤖"]:
    print(repr(sample), tokenizer.convert_ids_to_tokens(tokenizer.encode(sample)))

'1234567' ['1', '2', '3', '4', '5', '6', '7']
'1.234.567' ['1', '.', '2', '3', '4', '.', '5', '6', '7']
'antidesestabelecimento' ['ant', 'ides', 'est', 'abe', 'lec', 'imento']
'🤖' ['ðŁ¤ĸ']


O token é a unidade de custo e de limite, e a janela de contexto se mede nessa unidade. Conteúdo em português costuma ocupar mais tokens que o equivalente em inglês, porque o vocabulário foi ajustado sobre um corpus majoritariamente em inglês e código. A razão entre caracteres e tokens quantifica a diferença.

In [14]:
samples = [
    ("en", "Agents need tools to act in the world."),
    ("pt", "Agentes precisam de ferramentas para agir no mundo."),
]
pd.DataFrame(
    [
        {
            "kind": kind,
            "characters": len(sample),
            "tokens": len(tokenizer.encode(sample)),
            "chars_per_token": round(len(sample) / len(tokenizer.encode(sample)), 2),
        }
        for kind, sample in samples
    ]
)

,kind,characters,tokens,chars_per_token
0,en,38,9,4.22
1,pt,51,14,3.64


### Tokens especiais

Parte do vocabulário é reservada a marcadores de fronteira, acrescentados fora do algoritmo de fusão. O token de fim é o mecanismo pelo qual o modelo sinaliza o encerramento da geração, e na ausência dele a única parada seria o limite de tokens. O de preenchimento iguala o comprimento das sequências dentro de um lote, e parte dos modelos causais não o define.

In [15]:
print(f"eos_token: {tokenizer.eos_token} id {tokenizer.eos_token_id}")
print(f"pad_token: {tokenizer.pad_token} id {tokenizer.pad_token_id}")
print(f"tamanho do vocabulário: {len(tokenizer)}")

eos_token: <|endoftext|> id 151643
pad_token: <|endoftext|> id 151643
tamanho do vocabulário: 151665


## Forward pass e logits

Os logits são valores reais sem normalização. A distribuição sobre o vocabulário resulta do softmax,

$$p_i = \frac{e^{z_i}}{\sum_{j} e^{z_j}},$$

em que $z_i$ é o logit do token $i$. A função abaixo repete o forward pass da primeira seção e mostra os candidatos mais prováveis com as respectivas probabilidades.

In [16]:
def top_tokens(text: str, k: int = 5) -> pd.DataFrame:
    """Devolve os k tokens mais prováveis para a continuação de um texto."""
    encoded = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = base_model(**encoded).logits[0, -1]
    top = torch.topk(F.softmax(logits.float(), dim=-1), k)
    return pd.DataFrame(
        {
            "piece": [tokenizer.decode([token_id]) for token_id in top.indices.tolist()],
            "probability": [round(value, 4) for value in top.values.tolist()],
        }
    )

In [17]:
top_tokens(prompt)

,piece,probability
0,Paris,0.3156
1,______,0.1146
2,____,0.0635
3,__,0.0551
4,:\n,0.0513


A escolha feita na primeira seção corresponde à primeira linha da tabela, que reúne menos de um terço da massa de probabilidade. Os demais candidatos são lacunas de preenchimento, forma comum em material de exercício, e a dispersão observada se refere ao formato da continuação e não ao fato em si.

As duas células seguintes contrastam uma expressão fixa com uma continuação que admite muitas respostas legítimas.

In [18]:
top_tokens("Once upon a")

,piece,probability
0,time,0.9752
1,Time,0.0027
2,midnight,0.0014
3,summer,0.0009
4,night,0.0006


In [19]:
top_tokens("Her favorite color is")

,piece,probability
0,blue,0.3613
1,red,0.1498
2,purple,0.0642
3,green,0.0623
4,yellow,0.0470


A primeira concentra quase toda a massa em um único token. A segunda distribui a massa entre várias cores, todas plausíveis. A regra que converte essa distribuição em uma escolha é uma decisão de projeto, tratada nas duas seções seguintes.

## Geração

Gerar um texto completo aplica o ciclo da primeira seção repetidamente: ler os logits da última posição, escolher um id, anexá-lo à sequência e repetir. Duas decisões definem o resultado, e são tratadas nas duas subseções: como o ciclo é executado e como cada escolha é feita.

### Decodificação autorregressiva

O token escolhido em um passo compõe a entrada do passo seguinte, e por isso a sequência cresce a cada iteração.

In [20]:
input_ids = inputs["input_ids"]
for _ in range(8):
    with torch.no_grad():
        logits = base_model(input_ids).logits
    next_token = logits[0, -1].argmax()
    # A sequência cresce a cada passo e volta inteira para o modelo.
    input_ids = torch.cat([input_ids, next_token.view(1, 1)], dim=-1)
print(tokenizer.decode(input_ids[0]))

The capital of France is Paris. It is the largest city in


O método `generate` implementa esse laço, com duas vantagens sobre a versão acima: mantém em cache os estados intermediários de atenção, em vez de reprocessar a sequência inteira a cada passo, e encerra quando o modelo emite o token de fim. O argumento `do_sample=False` seleciona sempre o token de maior pontuação, regra chamada de decodificação greedy,

$$x_t = \arg\max_i P(x_t = i \mid x_{<t}).$$

In [21]:
generated = base_model.generate(**inputs, max_new_tokens=8, do_sample=False)
print(tokenizer.decode(generated[0]))

The capital of France is Paris. It is the largest city in


A saída coincide com a do laço manual. A célula seguinte usa o mesmo critério em um prompt de continuação aberta e com um orçamento maior de tokens. Antes de rodar, tente prever o que acontece.

In [22]:
open_prompt = "Three uses for a paperclip:"
open_inputs = tokenizer(open_prompt, return_tensors="pt").to(device)
generated = base_model.generate(**open_inputs, max_new_tokens=60, do_sample=False)
print(tokenizer.decode(generated[0]))

Three uses for a paperclip: 1. A paperclip is a small, round, metal object that can be used to hold paper together. 2. A paperclip is a small, round, metal object that can be used to hold paper together. 3. A paperclip is a small, round, metal object


O texto entra em repetição. O `argmax` é determinístico, e em continuações abertas ele realimenta o trecho que acabou de produzir, o que torna o mesmo trecho ainda mais provável. Esse fenômeno é a degeneração por repetição.

### Temperatura, top-k e top-p

A alternativa à escolha determinística é sortear o próximo id da distribuição, o que `do_sample=True` faz. Três argumentos de `generate` moldam a distribuição antes do sorteio, e todos operam sobre os logits daquele passo.

A temperatura $T$ reescala os logits antes do softmax,

$$p_i(T) = \frac{e^{z_i / T}}{\sum_{j} e^{z_j / T}},$$

de modo que $T < 1$ concentra a massa nos primeiros candidatos e $T > 1$ achata a curva, o que aumenta a chance de tokens improváveis. A função abaixo apenas encurta a comparação, gerando a continuação e devolvendo o texto novo.

In [23]:
def complete(text: str, max_new_tokens: int = 30, **options) -> str:
    """Gera uma continuação com o modelo base e devolve apenas o texto novo."""
    encoded = tokenizer(text, return_tensors="pt").to(device)
    generated = base_model.generate(**encoded, max_new_tokens=max_new_tokens, **options)
    return tokenizer.decode(generated[0][encoded["input_ids"].shape[-1] :]).strip()

In [24]:
for temperature in [0.2, 0.8, 1.5]:
    torch.manual_seed(7)
    print(f"temperature {temperature}")
    print(complete(open_prompt, do_sample=True, temperature=temperature))

temperature 0.2
1. to hold a pencil 2. to hold a pen 3. to hold a pencil case 4. to hold a pencil sharp
temperature 0.8
to hold letters, to fix a knot, to make a stapler, and to make a pencil. The number of uses for a paperclip is
temperature 1.5
to fast track someone’s conversation with you; to show respect to someone else when they just spoke before you; to show your displeased state over having


Os dois cortes restringem o conjunto de candidatos antes do sorteio, e eliminam a cauda de tokens que a temperatura alta tornou acessível. O `top_k` mantém os $k$ tokens de maior probabilidade, com $k$ fixo em todos os passos. O `top_p`, chamado de amostragem por núcleo, mantém o menor conjunto de tokens, ordenados por probabilidade decrescente, cuja soma alcança $p$; a quantidade de candidatos varia a cada passo, e é maior quando o modelo está indeciso.

Os valores padrão de `generate` deixam os dois cortes praticamente inativos neste modelo, com `top_k=50` e `top_p=1.0`. Esses padrões vêm da biblioteca e podem ser sobrescritos pelo `generation_config.json` publicado junto dos pesos, o que torna a configuração efetiva uma propriedade de cada modelo. A célula abaixo mantém a temperatura alta e acrescenta um corte de cada vez.

In [25]:
for cut in [{}, {"top_k": 5}, {"top_p": 0.9}]:
    torch.manual_seed(7)
    print(cut)
    print(complete(open_prompt, do_sample=True, temperature=1.5, **cut))

{}
to fast track someone’s conversation with you; to show respect to someone else when they just spoke before you; to show your displeased state over having
{'top_k': 5}
to fast track, to save time, to make a mess, and to be a joke. A paperclip is the perfect paperclip for a paper
{'top_p': 0.9}
to fast track someone’s conversation with you; to show respect to someone else when they make inappropriate use of your voice when you’re using someone else’s


O sorteio introduz aleatoriedade, e reproduzir um experimento passa a exigir semente fixada antes de cada geração, como nas duas células anteriores. Sem isso, a mesma configuração produz saídas diferentes a cada execução.

In [26]:
torch.manual_seed(7)
first = complete(open_prompt, do_sample=True, temperature=0.8)
torch.manual_seed(7)
second = complete(open_prompt, do_sample=True, temperature=0.8)
third = complete(open_prompt, do_sample=True, temperature=0.8)
print(first == second, first == third)

True False


A escolha depende do uso. Decisões que alimentam outro trecho de código, como selecionar uma ferramenta ou produzir JSON válido, pedem uma regra determinística e reprodutível. Gerar alternativas para comparação, ou texto sem repetição aparente, pede amostragem. Temperaturas altas degradam a saída com rapidez, com seleção de tokens pouco compatíveis com o contexto.

## Conversa e ajuste para instrução

Um problema permanece fora do alcance da regra de escolha. A célula abaixo submete uma pergunta direta ao modelo base. Antes de rodar, tente prever se a saída será uma resposta.

In [27]:
question = "What is the capital of Brazil?"
question_inputs = tokenizer(question, return_tensors="pt").to(device)
generated = base_model.generate(**question_inputs, max_new_tokens=40, do_sample=False)
print(tokenizer.decode(generated[0]))

What is the capital of Brazil? Brazil is a country located in South America. The capital of Brazil is Brasília, which is the capital city of the country. Brasília is located in the state of São Paulo, and it is


A resposta correta aparece no início, e a geração prossegue até o limite de tokens, repetindo a informação e acrescentando conteúdo incorreto. Prever o próximo token de documentos não inclui a noção de turno, e nada no texto cru indica ao modelo onde a resposta deveria terminar.

O ajuste para instrução acrescenta uma etapa de treino sobre diálogos, na qual o modelo aprende a responder e a encerrar o turno dentro de um formato específico de marcação. As células seguintes carregam a versão ajustada do mesmo modelo, de mesmo tamanho, mesma arquitetura e mesmo vocabulário.

In [28]:
INSTRUCT_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
chat_tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_NAME)
print(chat_tokenizer.chat_template is not None)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

True


In [29]:
chat_model = AutoModelForCausalLM.from_pretrained(
    INSTRUCT_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
chat_model.eval()
print(type(chat_model).__name__)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM


### Template de conversa

O formato é o template de conversa, distribuído junto do tokenizador e específico de cada família de modelos. A lista de dicionários com `role` e `content` é uma estrutura de conveniência, e o template a converte em uma única string.

In [30]:
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": question},
]
chat_prompt = chat_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(chat_prompt)

<|im_start|>system
You are a concise assistant.<|im_end|>
<|im_start|>user
What is the capital of Brazil?<|im_end|>
<|im_start|>assistant



Os marcadores `<|im_start|>` e `<|im_end|>` delimitam cada turno. O argumento `add_generation_prompt` acrescenta o cabeçalho do turno do assistente, sem o qual a geração continuaria a mensagem do usuário.

A geração usa a mesma chamada das seções anteriores, com o recorte do prompt para exibir apenas os tokens novos.

In [31]:
chat_inputs = chat_tokenizer(chat_prompt, return_tensors="pt").to(device)
generated = chat_model.generate(**chat_inputs, max_new_tokens=60, do_sample=False)
print(chat_tokenizer.decode(generated[0][chat_inputs["input_ids"].shape[-1] :]))

The capital of Brazil is Brasília.<|im_end|>


A resposta é direta e a geração encerra antes do limite de tokens.

### O prompt em tokens

A string acima é uma representação para leitura humana. O modelo recebe a sequência de ids dessa string, e é nela que os papéis e as fronteiras de turno existem. A conversão é o mesmo `encode` da seção de tokenização, porque o template produz texto comum.

In [32]:
chat_ids = chat_tokenizer.encode(chat_prompt)
pd.DataFrame(
    {
        "id": chat_ids,
        "token": chat_tokenizer.convert_ids_to_tokens(chat_ids),
    }
)

,id,token
0,151644,<|im_start|>
1,8948,system
2,198,Ċ
3,2610,You
4,525,Ġare
5,264,Ġa
6,63594,Ġconcise
7,17847,Ġassistant
8,13,.
9,151645,<|im_end|>


A tabela mostra a conversa como o modelo a enxerga. Os marcadores `<|im_start|>` e `<|im_end|>` ocupam ids próprios, acima de 151000, na faixa reservada aos tokens especiais. O `Ċ` é a quebra de linha que separa o cabeçalho do turno do seu conteúdo. As três últimas linhas são o efeito do `add_generation_prompt`, e a sequência termina no ponto exato em que a resposta deve começar.

Os nomes dos papéis são tokens comuns do vocabulário, sem qualquer status especial, e recebem os mesmos ids nos dois tokenizadores. A separação entre instrução do sistema e mensagem do usuário existe porque o treino associou esses rótulos a comportamentos distintos.

In [33]:
for token in ["<|im_start|>", "<|im_end|>", "<|endoftext|>", "system", "user", "assistant"]:
    print(token, chat_tokenizer.convert_tokens_to_ids(token), tokenizer.convert_tokens_to_ids(token))

<|im_start|> 151644 151644
<|im_end|> 151645 151645
<|endoftext|> 151643 151643
system 8948 8948
user 872 872
assistant 77091 77091


A diferença entre os dois tokenizadores está no token declarado como fim de sequência. No modelo base é o fim de documento, e no modelo instruído é o fim de turno, que é o que faz `generate` parar ao final da resposta.

In [34]:
print(f"base: {tokenizer.eos_token} id {tokenizer.eos_token_id}")
print(f"instruct: {chat_tokenizer.eos_token} id {chat_tokenizer.eos_token_id}")

base: <|endoftext|> id 151643
instruct: <|im_end|> id 151645


A marcação tem custo. O prompt final ocupa o dobro dos tokens do texto que as mensagens de fato carregam, e cada turno acrescenta uma quantidade fixa de marcadores.

In [35]:
content_tokens = sum(len(chat_tokenizer.encode(message["content"])) for message in messages)
print(f"conteúdo das mensagens: {content_tokens} tokens")
print(f"prompt com template: {len(chat_ids)} tokens")

conteúdo das mensagens: 13 tokens
prompt com template: 26 tokens


### Histórico da conversa

O template acomoda histórico. Turnos anteriores entram como mensagens e são convertidos na mesma string, de modo que o estado da conversa reside inteiramente na sequência enviada ao modelo, que cresce a cada rodada.

In [36]:
history = messages + [
    {"role": "assistant", "content": "Brasilia."},
    {"role": "user", "content": "And its population?"},
]
history_prompt = chat_tokenizer.apply_chat_template(
    history, tokenize=False, add_generation_prompt=True
)
print(history_prompt)

<|im_start|>system
You are a concise assistant.<|im_end|>
<|im_start|>user
What is the capital of Brazil?<|im_end|>
<|im_start|>assistant
Brasilia.<|im_end|>
<|im_start|>user
And its population?<|im_end|>
<|im_start|>assistant



In [37]:
print(f"prompt com dois turnos a mais: {len(chat_tokenizer.encode(history_prompt))} tokens")

prompt com dois turnos a mais: 44 tokens


Cada rodada reenvia a conversa inteira, e o custo de uma chamada cresce com o número de turnos acumulados.

In [38]:
history_inputs = chat_tokenizer(history_prompt, return_tensors="pt").to(device)
generated = chat_model.generate(**history_inputs, max_new_tokens=60, do_sample=False)
print(chat_tokenizer.decode(generated[0][history_inputs["input_ids"].shape[-1] :]))

The current population of Brasília is approximately 900,000 people.<|im_end|>


A pergunta seguinte foi respondida com apoio no turno anterior, ainda que o número esteja errado. A próxima célula envia exatamente a mesma string ao modelo base, cujo vocabulário contém os mesmos marcadores. Antes de rodar, tente prever a saída.

In [39]:
generated = base_model.generate(**history_inputs, max_new_tokens=60, do_sample=False)
print(chat_tokenizer.decode(generated[0][history_inputs["input_ids"].shape[-1] :]))

124.5 million.ponde
LETETERNS
LETETERNS
LETETERNS
LETETERNS
LETETERNS
LETETERNS
LETETERNS
LETETERNS
LETETERNS
LETETERNS
LETETERNS
LETETERNS
LETETERNS


O modelo base emite um número plausível e em seguida abandona o formato, com repetição até o limite de tokens. Os marcadores estão no vocabulário dos dois modelos, e o comportamento de assistente vem da etapa de treino que ensinou a respeitá-los.

## A classe LLM

As mesmas quatro operações se repetem a cada chamada: aplicar o template, tokenizar, gerar e recortar o prompt da resposta. A classe abaixo reúne essas operações em uma interface estável.

Duas decisões de projeto. A primeira é guardar tokenizador e pesos em um objeto, já que carregá-los é caro e eles sempre andam juntos; o restante do código da disciplina permanece em funções. A segunda é registrar tokens de entrada, tokens de saída e tempo a cada chamada, porque o custo e a latência de um sistema baseado em modelo se medem nessas unidades, e esse registro acompanha toda execução daqui em diante.

Temperatura zero desliga a amostragem, o que reproduz a decodificação greedy sem exigir um segundo argumento. O `chat` aplica o template de conversa e delega a geração, e o `invoke` encaminha string para um método e lista de mensagens para o outro, de modo que o restante do código não precise saber qual das duas formas foi escolhida.

In [40]:
class LLM:
    """Modelo de linguagem local, com tokenizador, geração e registro de uso."""

    def __init__(
        self,
        model: str,
        temperature: float = 0.7,
        top_p: float = 0.9,
        max_tokens: int = 256,
    ) -> None:
        self.model = model
        self.temperature = temperature
        self.top_p = top_p
        self.max_tokens = max_tokens
        self.tokenizer = AutoTokenizer.from_pretrained(model)
        self.weights = AutoModelForCausalLM.from_pretrained(
            model, dtype=torch.float16 if device == "cuda" else torch.float32
        ).to(device)
        self.weights.eval()
        # Nem todo tokenizador define token de preenchimento; o de fim serve.
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.usage: list[dict] = []

    def generate(
        self,
        prompt: str,
        temperature: float | None = None,
        max_tokens: int | None = None,
    ) -> str:
        """Gera texto a partir de uma string, sem aplicar template de conversa."""
        temperature = self.temperature if temperature is None else temperature
        max_tokens = self.max_tokens if max_tokens is None else max_tokens
        sampling = temperature > 0

        inputs = self.tokenizer(prompt, return_tensors="pt").to(device)
        started = time.perf_counter()
        with torch.no_grad():
            output = self.weights.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=sampling,
                temperature=temperature if sampling else None,
                top_p=self.top_p if sampling else None,
                pad_token_id=self.tokenizer.pad_token_id,
            )
        new_tokens = output[0][inputs["input_ids"].shape[-1] :]

        self.usage.append(
            {
                "tokens_in": int(inputs["input_ids"].shape[-1]),
                "tokens_out": int(new_tokens.shape[-1]),
                "seconds": round(time.perf_counter() - started, 3),
            }
        )
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def chat(self, messages: list[dict], **kwargs) -> str:
        """Aplica o template de conversa às mensagens e gera a resposta."""
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        return self.generate(prompt, **kwargs)

    def invoke(self, input: str | list[dict], **kwargs) -> str:
        """Encaminha string para generate e lista de mensagens para chat."""
        if isinstance(input, str):
            return self.generate(input, **kwargs)
        return self.chat(input, **kwargs)

In [41]:
llm = LLM(INSTRUCT_NAME, temperature=0.0, max_tokens=64)
print(llm.invoke(messages))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The capital of Brazil is Brasília.


In [42]:
print(llm.invoke(history))

The current population of Brasília is approximately 900,000 people.


O registro acumulado quantifica o custo de uma sequência de chamadas. A taxa de tokens por segundo mede a latência de forma independente do tamanho da resposta.

In [43]:
usage = pd.DataFrame(llm.usage)
usage["tokens_per_second"] = (usage["tokens_out"] / usage["seconds"]).round(1)
usage

,tokens_in,tokens_out,seconds,tokens_per_second
0,26,9,2.074,4.3
1,44,19,4.849,3.9


Essa classe corresponde, com pequenas diferenças, à `LLM` de `agentkit/model.py`, que passa a ser a via de acesso ao modelo.

## Exercícios

### Exercício 1

Escreva uma frase de mais ou menos vinte palavras e a mesma frase traduzida para o inglês. Conte os tokens das duas e diga qual delas sairia mais cara em uma chamada.

In [47]:
sentence_pt = "Quais as profissões mais eminentes com o avanço da IA para o futuro e quais que possuem risco de extinguir?"
sentence_en = "Which professions are most prominent as AI advances into the future, and which ones are at risk of extinction?"


print(len(tokenizer.encode(sentence_pt)))
print(len(tokenizer.encode(sentence_en)))



31
21


### Exercício 2

Escreva três frases incompletas, uma em que a continuação pareça óbvia e duas em que pareça incerta, e anote a ordem esperada antes de rodar. Passe as três por `top_tokens` e ordene pela probabilidade do primeiro candidato.

In [73]:
prefixes = ["To be or not to be that id the", "knock", "my god why have you"]
##Question, knock, forseake
for prefixe in prefixes:
    resultado = top_tokens(prefixe)
    print(resultado.iloc[0].piece)

 question
out
 been


### Exercício 3

Gere uma continuação com `do_sample=True` e `top_k=1`, e outra com `do_sample=False`, a partir do mesmo prompt. Explique em uma frase por que as duas saídas coincidem.

In [76]:
prompt = "A maior rodovia do Brasil é"
outputSample = complete(prompt, do_sample=True, top_k=1)
outputNotSample = complete(prompt, do_sample=False, top_k=1)
print(outputSample)
print(outputNotSample)
##Só possui 1 k, desta forma, só existe uma escolha.

a Avenida Brasil, que passa pela cidade de São Paulo. A Avenida Brasil é uma das mais importantes rodovias do país, pass
a Avenida Brasil, que passa pela cidade de São Paulo. A Avenida Brasil é uma das mais importantes rodovias do país, pass


### Exercício 4

Troque a mensagem de sistema por uma que peça respostas em português, refaça a pergunta ao modelo instruído e conte quantos tokens o prompt ganhou com a troca.

In [ ]:
new_messages = []

### Exercício 5

Faça três perguntas com a classe `LLM` e use o registro em `usage` para dizer qual resposta gerou mais tokens e qual foi a mais rápida em tokens por segundo.

In [ ]:
questions = []